# 04 - Deep Learning opcional: red neuronal para ventas

Este notebook prueba una red neuronal multicapa sencilla para el mismo problema de prediccion de ventas.

Logica de negocio:

- No usamos deep learning porque suene avanzado, sino para comprobar si aporta valor real.
- Si no mejora a modelos tabulares mas simples, la conclusion defendible es quedarse con el modelo mas interpretable.
- Para este TFM basta una prueba razonable y explicada; no conviene convertir el proyecto en un TFM de deep learning puro.

## 0. Preparacion

Usamos `MLPRegressor` de scikit-learn como red neuronal feed-forward. Es ligero y suficiente para comparar complejidad frente a modelos clasicos.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from conexion_sql import read_sql
from eda_utils import asegurar_directorio
from modelos_utils import (
    crear_features_temporales,
    crear_lags_por_grupo,
    metricas_regresion,
    preparar_resultados_prediccion,
    train_test_temporal,
)
from visualizacion_utils import guardar_figura, grafico_real_vs_predicho

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

OUT_DATOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "datos")
OUT_GRAFICOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "graficos")

## 1. Preparacion del dataset

Usamos exactamente el mismo dataset que en ML para comparar de forma justa.

In [ ]:
df = read_sql("SELECT * FROM dbo.vw_prediccion_base_ventas")
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values(["canal", "categoria", "fecha"]).reset_index(drop=True)

df_features = crear_features_temporales(df, "fecha")
df_features = crear_lags_por_grupo(
    df_features,
    grupo_cols=["canal", "categoria"],
    objetivo="unidades_vendidas",
    lags=[1, 7, 14, 30],
    ventanas=[7, 14, 30],
)
df_modelo = df_features.dropna().copy()

train, test = train_test_temporal(df_modelo, "fecha", test_ratio=0.2)

objetivo = "unidades_vendidas"
features_numericas = [
    "anio",
    "mes_numero",
    "dia_semana",
    "es_fin_de_semana",
    "dias_desde_inicio",
    "media_movil_7d_unidades",
    "media_movil_30d_unidades",
    "unidades_vendidas_lag_1",
    "unidades_vendidas_lag_7",
    "unidades_vendidas_lag_14",
    "unidades_vendidas_lag_30",
    "unidades_vendidas_media_7",
    "unidades_vendidas_media_14",
    "unidades_vendidas_media_30",
]
features_categoricas = ["canal", "categoria"]
features = features_numericas + features_categoricas

X_train = train[features]
y_train = train[objetivo]
X_test = test[features]
y_test = test[objetivo]

print(X_train.shape, X_test.shape)

## 2. Red neuronal MLP

La red tiene varias capas ocultas. Activamos `early_stopping` para detener entrenamiento si no mejora, reduciendo sobreajuste.

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), features_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas),
    ]
)

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,
)

modelo_dl = Pipeline([("preprocesador", preprocesador), ("modelo", mlp)])
modelo_dl.fit(X_train, y_train)

pred_dl = np.clip(modelo_dl.predict(X_test), 0, None)
metricas_dl = pd.DataFrame([{ "modelo": "MLP Neural Network", **metricas_regresion(y_test, pred_dl)}])
display(metricas_dl)
metricas_dl.to_csv(OUT_DATOS / "metricas_deep_learning_mlp.csv", index=False)

predicciones_dl = preparar_resultados_prediccion(test[["fecha", "canal", "categoria"]], y_test, pred_dl, "MLP Neural Network")
predicciones_dl.to_csv(OUT_DATOS / "predicciones_deep_learning_mlp.csv", index=False)

## 3. Comparacion con machine learning clasico

Si ya se ejecuto el notebook 03, cargamos sus metricas para comparar.

In [ ]:
ruta_metricas_ml = OUT_DATOS / "metricas_ml_ventas.csv"

if ruta_metricas_ml.exists():
    metricas_ml = pd.read_csv(ruta_metricas_ml)
    comparacion = pd.concat([metricas_ml, metricas_dl], ignore_index=True).sort_values("RMSE")
else:
    comparacion = metricas_dl.copy()

display(comparacion)
comparacion.to_csv(OUT_DATOS / "comparacion_ml_deep_learning.csv", index=False)

## 4. Grafico real vs predicho

In [ ]:
serie_pred_dl = predicciones_dl.groupby("fecha", as_index=False).agg(real=("real", "sum"), prediccion=("prediccion", "sum"))
fig, ax = grafico_real_vs_predicho(serie_pred_dl, titulo="Unidades reales vs predichas - MLP")
guardar_figura("deep_learning_mlp_real_vs_predicho.png", OUT_GRAFICOS)
plt.show()

## 5. Curva de entrenamiento

Sirve para explicar si la red aprendio o si se quedo estancada.

In [ ]:
loss_curve = pd.DataFrame({"iteracion": range(1, len(modelo_dl.named_steps["modelo"].loss_curve_) + 1), "loss": modelo_dl.named_steps["modelo"].loss_curve_})
display(loss_curve.tail())

plt.figure(figsize=(10, 4))
sns.lineplot(data=loss_curve, x="iteracion", y="loss")
plt.title("Curva de perdida - MLP")
guardar_figura("deep_learning_mlp_loss_curve.png", OUT_GRAFICOS)
plt.show()

## 6. Decision de negocio

Completar despues de ejecutar:

- Si la red neuronal mejora claramente al mejor modelo clasico, se puede defender como aporte avanzado.
- Si no mejora, se defiende que el problema se resuelve mejor con modelos tabulares interpretables.
- En ambos casos, el valor de negocio esta en anticipar demanda para stock, promociones y margen.